# Análisis de video y estado del tráfico

Workflow de inferencia de VAAET ML 4.5.3. Procesa minutos completos, genera video anotado, ejecuta el clasificador jerárquico y permite persistencia y revisión HITL opcionales.

> Los flags de este notebook autorizan un flujo de uso; **no promocionan modelos** ni modifican el manifiesto del bundle.

## 1. Configuración central

Editá solamente la siguiente celda antes de ejecutar el entorno. Las demás celdas consumen estas opciones sin redefinirlas. Los valores predeterminados ejecutan un piloto conservador, sin PostgreSQL ni revisión humana.

In [ ]:
# Workflow configuration — edit only this cell
ALLOW_PILOT_BUNDLE = True
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = False
ENABLE_HUMAN_REVIEW = False
REVIEW_MODE = "priority"  # priority or all
DOWNLOAD_ANNOTATED_VIDEO = True
SHOW_DASHBOARD = True
HUD_DEBUG = False

if REVIEW_MODE not in {"priority", "all"}:
    raise ValueError("REVIEW_MODE must be 'priority' or 'all'.")

print("✅ Workflow configuration validated")
print(f"   PostgreSQL persistence: {'enabled' if PERSIST_TO_DATABASE else 'disabled'}")
print(f"   Human review: {REVIEW_MODE if ENABLE_HUMAN_REVIEW else 'disabled'}")
print(f"   Annotated video download: {'enabled' if DOWNLOAD_ANNOTATED_VIDEO else 'disabled'}")
print(f"   Dashboard: {'enabled' if SHOW_DASHBOARD else 'disabled'}")
print(f"   HUD mode: {'technical debug' if HUD_DEBUG else 'public'}")

### Guía de escenarios

Todos los escenarios requieren un MP4 y un bundle completo de cuatro archivos. En Colab, el bundle se busca en este orden: `artifacts/traffic-state/` local, Google Drive y upload manual. `HUD_DEBUG=False` produce el video público; `True` añade IDs, confianza, evidencia y calidad para diagnóstico.

Los escenarios con PostgreSQL reutilizan el endpoint común y solicitan únicamente los perfiles `inference` o `review` que correspondan. La configuración completa de Secrets y TLS vive en la [guía canónica de Colab](../../docs/operations/colab-guide.md#secrets-y-postgresql). Tener credenciales disponibles no activa ninguna escritura: mandan los flags de la celda inicial.

#### A. Inferencia piloto rápida

```python
ALLOW_PILOT_BUNDLE = True
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = False
ENABLE_HUMAN_REVIEW = False
REVIEW_MODE = "priority"
```

**Entrada:** bundle semilla y video. **Secrets:** ninguno. **Salida:** video anotado y `df_classified` en memoria. **Siguiente paso:** inspeccionar resultados o activar HITL.

#### B. Piloto con HITL portable, sin PostgreSQL

```python
ALLOW_PILOT_BUNDLE = True
PERSIST_TO_DATABASE = False
ENABLE_HUMAN_REVIEW = True
REVIEW_MODE = "priority"
```

**Entrada:** bundle piloto y video de al menos dos minutos consecutivos. **Secrets:** `VAAET_REVIEWER_ID`. **Salida:** formulario de revisión y paquete HITL inmutable. **Siguiente paso:** revisar las filas y ejecutar `finalize_current_review()`. En Colab se intenta sincronizar con Drive.

#### C. Persistencia operacional sin revisión

```python
PERSIST_TO_DATABASE = True
ENABLE_HUMAN_REVIEW = False
```

**Entrada:** bundle piloto o de producción y video. **Perfil:** `inference`. **Salida:** features y predicciones idempotentes en PostgreSQL. **Siguiente paso:** habilitar revisión en otra ejecución si se necesita feedback.

#### D. PostgreSQL con revisión prioritaria

```python
PERSIST_TO_DATABASE = True
ENABLE_HUMAN_REVIEW = True
REVIEW_MODE = "priority"
```

**Entrada:** bundle piloto o de producción y video. **Perfiles:** `inference` y `review`, más la identidad del revisor. **Salida:** features, predicciones, validaciones append-only y paquete HITL de la sesión. **Siguiente paso:** terminar la cola y ejecutar `finalize_current_review()`.

#### E. Revisión completa de un clip

```python
ENABLE_HUMAN_REVIEW = True
REVIEW_MODE = "all"
```

**Entrada:** cualquier ejecución con minutos clasificados. **Secrets:** `VAAET_REVIEWER_ID` y, si corresponde, perfil `review`. **Salida:** una decisión por cada minuto revisado. `all` muestra todos los minutos completos; `priority` selecciona candidatos de incidente, baja confianza y transiciones. **Siguiente paso:** finalizar la sesión; las filas omitidas nunca son ground truth.

#### F. Candidato experimental

```python
ALLOW_PILOT_BUNDLE = False
ALLOW_EXPERIMENTAL_BUNDLE = True
PERSIST_TO_DATABASE = False
ENABLE_HUMAN_REVIEW = False
```

**Entrada:** bundle `candidate` y video de evaluación. **Secrets:** ninguno, salvo `VAAET_REVIEWER_ID` si activás una revisión portable. **Salida:** resultados offline y, opcionalmente, paquete HITL local/Drive. **Siguiente paso:** comparar métricas fuera de este notebook; la autorización no lo vuelve `production` y la persistencia operacional está bloqueada.

#### G. Bundle aprobado para producción

```python
ALLOW_PILOT_BUNDLE = False
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = True
ENABLE_HUMAN_REVIEW = True
REVIEW_MODE = "priority"
```

**Entrada:** bundle cuyo manifiesto declara `deployment_stage=production` y video. **Secrets:** perfiles `inference`/`review` según los flags elegidos. **Salida:** video, predicciones y feedback operacional. **Siguiente paso:** monitorear resultados y cerrar cualquier sesión HITL; PostgreSQL y revisión continúan siendo decisiones independientes.

#### H. Clip corto o con un solo minuto completo

**Entrada:** MP4 menor a 120 segundos. **Secrets:** ninguno. **Salida:** siempre video anotado; con 60–119 segundos también se conserva una fila de telemetría, pero todavía no existe una clasificación estable porque las features temporales necesitan dos minutos consecutivos. **Siguiente paso:** usar un clip superior a dos minutos para observar estado, persistencia e histéresis. No requiere una configuración especial.

## 2. Preparar el entorno y cargar el bundle

La celda instala VAAET, valida el origen del paquete y busca el bundle localmente, en Google Drive o mediante upload. Luego verifica manifiesto, checksums, las 19 features, tres salidas MLP y los cuatro estados públicos.

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("VAAET repository root not found")

os.chdir(REPO_ROOT)

def validate_runtime_version(version: tuple[int, int]) -> None:
    if not (3, 10) <= version <= (3, 13):
        raise RuntimeError(
            f"Unsupported Python {version[0]}.{version[1]}. VAAET supports Python 3.10–3.13. "
            "In Colab, select a compatible runtime such as 2026.07 (Python 3.12.13)."
        )

def install_project(command: list[str], *, extras: str) -> None:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode == 0:
        print(f"✅ VAAET installation completed | extras={extras}")
        return
    print("🔴 VAAET installation failed")
    print("----- pip stdout -----")
    print(result.stdout.strip() or "(empty)")
    print("----- pip stderr -----")
    print(result.stderr.strip() or "(empty)")
    raise RuntimeError(
        f"VAAET installation failed under Python {sys.version.split()[0]} with extras={extras}. "
        "Update the repository and re-run this cell. In Colab, runtime 2026.07 "
        "(Python 3.12.13) is the temporary fallback."
    )

CURRENT_PYTHON = (sys.version_info.major, sys.version_info.minor)
validate_runtime_version(CURRENT_PYTHON)
print(f"🐍 Runtime Python {sys.version.split()[0]} | supported: 3.10–3.13")
WORKFLOW_EXTRAS = "vision,training,visualization,database"
project_requirement = f"{REPO_ROOT}[{WORKFLOW_EXTRAS}]"
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(project_requirement)
else:
    install_command.extend(["-e", project_requirement])
install_project(install_command, extras=WORKFLOW_EXTRAS)

for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import vaaet

def validate_vaaet_origin(package: object, repo_root: Path, in_colab: bool) -> Path:
    package_file = getattr(package, "__file__", None)
    if not package_file:
        package_path = list(getattr(package, "__path__", ()))
        raise ImportError(
            "The 'vaaet' import resolved to a namespace package instead of the installed package. "
            f"Resolved locations: {package_path}. Re-run this setup cell."
        )
    origin = Path(package_file).resolve()
    expected_editable_root = (repo_root / "src/vaaet").resolve()
    if in_colab and repo_root.resolve() in origin.parents:
        raise ImportError(f"Colab must load the installed wheel, not repository path: {origin}")
    if not in_colab and origin.parent != expected_editable_root:
        raise ImportError(f"Local editable install has unexpected origin: {origin}")
    return origin

VAAET_PACKAGE_FILE = validate_vaaet_origin(vaaet, REPO_ROOT, IN_COLAB)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
pip_check_output = "\n".join(
    part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
)
if pip_check.returncode == 0:
    print("✅ pip check: no broken requirements found")
else:
    print("⚠️ pip check detected conflicts in the managed notebook runtime:")
    print(pip_check_output or "No diagnostic output was returned")
    print("ℹ️ Continuing because workflow imports are validated explicitly below.")

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not required"

print({name: package_version(name) for name in ("numpy", "tensorflow", "opencv-python-headless", "ultralytics-opencv-headless")})

import os
import shutil

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf
import ultralytics

from vaaet.artifacts import MANIFEST_FILE, validate_manifest
from vaaet.data.database import DatabaseProfile, database_engine, get_optional_database_settings, inspect_database, load_reviewer_id
from vaaet.data.persistence import persist_classified_telemetry
from vaaet.data.pipeline_runs import PipelineRunMetadata, PipelineWorkflow, pipeline_run
from vaaet.data.review import build_review_widget, finalize_review_session, load_review_queue, persist_human_validation, select_review_queue
from vaaet.inference.traffic_state import classify_raw_telemetry
from vaaet.logging import configure_logging
from vaaet.settings import DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABEL_MAP_PATH, MODEL_DIR, MODEL_PATH, RANDOM_SEED, SCALER_PATH, STATE_LABELS
from vaaet.vision.analysis import TrafficStatePrediction, analyze_video
from vaaet.vision.hud import HudConfig

configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | TensorFlow {tf.__version__} | GPU {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Package: {VAAET_PACKAGE_FILE}")
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f"✅ inference workflow ready | root={REPO_ROOT} | commit={GIT_COMMIT}")

_model_dir_abs = REPO_ROOT / MODEL_DIR
_model_dir_abs.mkdir(parents=True, exist_ok=True)
_ARTIFACT_NAMES = [Path(MODEL_PATH).name, Path(SCALER_PATH).name, Path(LABEL_MAP_PATH).name, MANIFEST_FILE]

def _bundle_paths(directory: Path) -> dict[str, Path]:
    return {name: directory / name for name in _ARTIFACT_NAMES}

paths = _bundle_paths(_model_dir_abs)
source = "local"
if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_paths = _bundle_paths(Path("/content/drive") / DRIVE_ARTIFACT_DIR)
        if all(path.is_file() for path in drive_paths.values()):
            for name, path in drive_paths.items():
                shutil.copy2(path, paths[name])
            source = "Google Drive"
    except Exception as exc:
        print(f"Drive unavailable: {exc}")

if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    from google.colab import files

    print("Upload the complete four-file model bundle:", _ARTIFACT_NAMES)
    for name, content in files.upload().items():
        if name in _ARTIFACT_NAMES:
            paths[name].write_bytes(content)
    source = "upload"

if all(path.is_file() for path in paths.values()):
    manifest = validate_manifest(_model_dir_abs)
    DEPLOYMENT_STAGE = manifest["training_lifecycle"]["deployment_stage"]
    MODEL_INPUT_POLICY = manifest["training_lifecycle"]["input_policy"]
    if DEPLOYMENT_STAGE == "pilot" and not ALLOW_PILOT_BUNDLE:
        raise RuntimeError("Pilot bundle loading is disabled. Set ALLOW_PILOT_BUNDLE=True explicitly.")
    if DEPLOYMENT_STAGE == "candidate" and not ALLOW_EXPERIMENTAL_BUNDLE:
        raise RuntimeError("Candidate bundle is not approved. Enable it only for explicit offline evaluation.")
    if DEPLOYMENT_STAGE == "candidate" and PERSIST_TO_DATABASE:
        raise RuntimeError("Candidate bundles are offline-only. Set PERSIST_TO_DATABASE=False.")
    model = tf.keras.models.load_model(paths[Path(MODEL_PATH).name])
    scaler = joblib.load(paths[Path(SCALER_PATH).name])
    label_mapping = joblib.load(paths[Path(LABEL_MAP_PATH).name])
    if dict(label_mapping) != dict(STATE_LABELS):
        raise RuntimeError("Bundle label_mapping.joblib does not match the four public states.")
    if int(model.output_shape[-1]) != 3:
        raise RuntimeError("Bundle MLP must expose exactly three stable-state outputs.")
    if int(getattr(scaler, "n_features_in_", -1)) != len(FEATURE_COLS):
        raise RuntimeError("Bundle scaler does not match the 19-feature contract.")
    hitl_destination = (
        "PostgreSQL + immutable package"
        if ENABLE_HUMAN_REVIEW and PERSIST_TO_DATABASE
        else "portable immutable package"
        if ENABLE_HUMAN_REVIEW
        else "disabled"
    )
    print(f"✅ Valid {DEPLOYMENT_STAGE.upper()} bundle loaded from {source} | input_policy={MODEL_INPUT_POLICY}")
    print("\n🧭 Active inference flow")
    print(f"   Bundle stage: {DEPLOYMENT_STAGE}")
    print(f"   PostgreSQL: {'enabled' if PERSIST_TO_DATABASE else 'disabled'}")
    print(f"   Human review: {REVIEW_MODE if ENABLE_HUMAN_REVIEW else 'disabled'}")
    print(f"   HITL destination: {hitl_destination}")
    if DEPLOYMENT_STAGE == "pilot":
        print("   ⚠️ Pilot authorization is active; this bundle is not production-approved.")
    elif DEPLOYMENT_STAGE == "candidate":
        print("   ⚠️ Experimental candidate authorization is active; operational persistence is blocked.")
else:
    missing = [name for name, path in paths.items() if not path.is_file()]
    raise FileNotFoundError(f"Incomplete model bundle; missing: {missing}")


## 3. Seleccionar el clip

En Colab se solicita un MP4 mediante upload. En desarrollo local se intenta usar `data/sample/sample.mp4`. El nombre recomendado es `bridge_YYYY-MM-DD_HH-MM-SS_to_HH-MM-SS.mp4` para conservar trazabilidad temporal.

In [ ]:
VIDEO_PATH: Path | None = None
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded))).resolve()
else:
    candidate = REPO_ROOT / "data/sample/sample.mp4"
    VIDEO_PATH = candidate if candidate.is_file() else None

print(f"Clip: {VIDEO_PATH or 'not selected'}")

## 4. Analizar, anotar y clasificar

El video se procesa completo y el contrato produce una fila raw por cada ventana completa de 60 segundos. El primer minuto es la línea base de las features temporales: se necesitan dos minutos consecutivos para obtener la primera fila clasificable. La clasificación final ya se ejecuta en esta celda; no es necesario volver a calcularla después.

In [ ]:
def prediction_provider(raw: pd.DataFrame) -> TrafficStatePrediction | None:
    try:
        classified = classify_raw_telemetry(
            raw,
            model,
            scaler,
            label_mapping=label_mapping,
            feature_cols=FEATURE_COLS,
            model_version=manifest["model_version"],
            input_policy=MODEL_INPUT_POLICY,
            inference_mode="stable",
            decision_policy=manifest["decision_policy"],
        )
    except ValueError:
        return None
    if classified.empty:
        return None
    latest = classified.iloc[-1]
    return TrafficStatePrediction(
        state=int(latest["traffic_state"]),
        label=str(latest["state_label"]),
        confidence=float(latest["confidence"]),
        evidence=float(latest.get("accident_evidence_score", 0.0)),
        incident_candidate=bool(latest.get("accident_rule_triggered", False)),
    )

if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Select or upload a valid MP4 clip before continuing")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4")
_run_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.INFERENCE, git_commit=GIT_COMMIT, source_kind="video", clip_id=VIDEO_PATH.stem, model_version=manifest["model_version"])
with pipeline_run(_run_metadata, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs") as _run:
    analysis_result = analyze_video(
        VIDEO_PATH,
        OUTPUT_VIDEO,
        prediction_provider=prediction_provider,
        hud_config=HudConfig(debug=HUD_DEBUG),
    )
    _run.set_output_rows(len(analysis_result.telemetry))
LOCAL_INFERENCE_RUN_ID = str(_run.id)
df_telemetry = analysis_result.telemetry
df_classified = None
INFERENCE_PIPELINE_RUN_ID = None
REVIEW_VALIDATIONS = []
REVIEW_EXPORT_FRAME = None
if df_telemetry.empty:
    print("ℹ️ The clip was analyzed successfully but contains no complete 60-second window.")
    print("   Feature engineering, classification, persistence, and HITL review were skipped.")
    print(f"   Processed: {analysis_result.processed_duration_seconds:.1f}s | minimum required: 60.0s")
else:
    df_classified = classify_raw_telemetry(
        df_telemetry,
        model,
        scaler,
        label_mapping=label_mapping,
        feature_cols=FEATURE_COLS,
        model_version=manifest["model_version"],
        input_policy=MODEL_INPUT_POLICY,
        inference_mode="stable",
        decision_policy=manifest["decision_policy"],
    )
    if df_classified.empty:
        print("ℹ️ The clip produced raw telemetry but not enough temporal context for a stable state.")
        print("   Stable classification requires at least two consecutive complete 60-second windows.")
        print("   Metrics, PostgreSQL persistence, HITL review, and dashboard were skipped.")
    else:
        display(df_classified)
        print("✅ Stable-state classification:")
        for code in sorted(df_classified["traffic_state"].unique()):
            count = int(df_classified["traffic_state"].eq(code).sum())
            print(f"   {STATE_LABELS.get(int(code), 'Unknown'):>10}: {count} classified minutes")
        automatic_accidents = int(df_classified["traffic_state"].eq(3).sum())
        incident_candidates = int(
            df_classified.get("accident_alert_started", pd.Series(False, index=df_classified.index)).sum()
        )
        print(f"   Automatic Accident states: {automatic_accidents} (must always be zero)")
        print(f"   Possible-incident candidates: {incident_candidates} (state remains Congested)")
    print(f"✅ Complete telemetry minutes: {analysis_result.complete_minutes} | classified rows: {len(df_classified)} | discarded tail: {analysis_result.discarded_partial_seconds:.1f}s")
print(f"✅ Annotated video: {analysis_result.video_path}")

if IN_COLAB and DOWNLOAD_ANNOTATED_VIDEO:
    from google.colab import files

    files.download(str(analysis_result.video_path))
elif IN_COLAB:
    print("ℹ️ Automatic video download disabled; the annotated file remains under /content.")

### Resultado de la clasificación

La celda anterior ya calculó `df_classified` mediante la cadena compartida `features → scaler → MLP → calibración → política temporal → detector de posible incidente`. No vuelvas a clasificar manualmente el mismo clip.

`traffic_state` sólo puede ser `Normal`, `Reduced` o `Congested`. Un posible accidente conserva `Congested` y activa `accident_rule_triggered`; `Accident` únicamente puede surgir de una validación humana.

## 5. Persistencia PostgreSQL opcional

Al activarla, el perfil `inference` escribe únicamente features y predicciones. Si la conexión, migración o permisos fallan, `df_classified` permanece disponible y el video no se pierde.

In [ ]:
# Cell 5 — Optional PostgreSQL persistence
inference_settings = (
    get_optional_database_settings(DatabaseProfile.INFERENCE) if PERSIST_TO_DATABASE else None
)

if inference_settings is not None:
    try:
        if df_classified is not None and not df_classified.empty:
            with database_engine(inference_settings) as db_engine:
                health = inspect_database(db_engine, DatabaseProfile.INFERENCE)
                print(f"PostgreSQL {health.server_version} | TLS={health.ssl_enabled} | schemas={health.available_schemas}")
                persisted = persist_classified_telemetry(
                    df_classified, engine=db_engine, model_version=manifest["model_version"]
                )
            INFERENCE_PIPELINE_RUN_ID = persisted.pipeline_run_id
            print(f"✅ Persistence completed: {persisted.telemetry_rows} feature rows | {persisted.classification_rows} predictions | run={INFERENCE_PIPELINE_RUN_ID}")
        else:
            print("ℹ️ Persistence skipped: no classified complete-minute rows are available.")
    except NameError:
        print("⚠️ Run the video analysis cell first, then re-run this cell.")
    except Exception as error:
        print(f"🔴 Persistence failed ({type(error).__name__}). Verify migration, TLS and role grants.")
        print("   Classified data is available in-memory (df_classified)")
elif PERSIST_TO_DATABASE:
    print("⚠️ PostgreSQL persistence was requested, but the inference profile is unavailable.")
    print("   Configure the read/write inference profile according to the canonical Colab guide.")
    print("   Classified rows remain safely available in df_classified.")
else:
    print("ℹ️ PostgreSQL persistence is disabled by the central configuration.")
    print("   Classified rows remain in df_classified and can still enter a portable HITL review.")


## 6. Revisión humana HITL

La revisión se abre **después** de terminar el clip; no interrumpe la inferencia con pop-ups. Mirá el video anotado, buscá el `record_time` indicado y contrastá también los minutos anterior y posterior.

- **Confirmar:** conservar el estado predicho.
- **Corregir:** elegir `Normal`, `Reduced` o `Congested`.
- **Omitir:** la fila queda pendiente y nunca se usa como ground truth.
- **Confirmar Accident:** elegir `Accident`, marcar `I reviewed temporal context` y escribir una nota. El estado automático permanece `Congested`; Accident sólo existe como decisión humana.

Con PostgreSQL, `Save validation` inserta una decisión append-only en `vaaet_feedback.human_validations`; nunca edites la predicción manualmente. Sin PostgreSQL, las decisiones quedan en el paquete portable. En ambos casos, al terminar ejecutá `finalize_current_review()` para cerrar la sesión de forma idempotente. El reentrenamiento se realiza exclusivamente en `train_traffic_state_classifier.ipynb`.


In [ ]:
# Cell 6 — Explicit human review after inference
if ENABLE_HUMAN_REVIEW:
    reviewer_id = load_reviewer_id()
    review_settings = get_optional_database_settings(DatabaseProfile.REVIEW)
    REVIEW_VALIDATIONS = []
    if review_settings is not None and INFERENCE_PIPELINE_RUN_ID is not None:
        full_review_queue = load_review_queue(
            settings=review_settings, pipeline_run_id=INFERENCE_PIPELINE_RUN_ID, mode="all"
        )
        review_queue = select_review_queue(full_review_queue, mode=REVIEW_MODE)
        print(f"✅ Database-backed review queue: {len(review_queue)} of {len(full_review_queue)} rows selected ({REVIEW_MODE}).")
        _prediction_keys = full_review_queue[["clip_id", "record_time", "prediction_id"]].copy()
        _prediction_keys["record_time"] = pd.to_datetime(_prediction_keys["record_time"], utc=True)
        REVIEW_EXPORT_FRAME = df_classified.copy()
        REVIEW_EXPORT_FRAME["record_time"] = pd.to_datetime(REVIEW_EXPORT_FRAME["record_time"], utc=True)
        REVIEW_EXPORT_FRAME = REVIEW_EXPORT_FRAME.merge(_prediction_keys, on=["clip_id", "record_time"], how="left", validate="one_to_one")
        if REVIEW_EXPORT_FRAME["prediction_id"].isna().any():
            raise RuntimeError("PostgreSQL review queue does not cover every inference row.")
        def _persist_and_accumulate(decision):
            persist_human_validation(decision, settings=review_settings)
            REVIEW_VALIDATIONS.append(decision)
        build_review_widget(review_queue, reviewer_id=reviewer_id, on_submit=_persist_and_accumulate)
    elif df_classified is not None and not df_classified.empty:
        reason = (
            "the inference run was not persisted"
            if INFERENCE_PIPELINE_RUN_ID is None
            else "the review database profile is unavailable"
        )
        print(f"ℹ️ Portable review mode: {reason}.")
        print("   Decisions will not modify PostgreSQL and will remain in the immutable session package.")
        REVIEW_EXPORT_FRAME = df_classified.copy().reset_index(drop=True)
        REVIEW_EXPORT_FRAME["prediction_id"] = REVIEW_EXPORT_FRAME.index + 1
        local_queue = select_review_queue(REVIEW_EXPORT_FRAME, mode=REVIEW_MODE)
        print(f"✅ Portable review queue: {len(local_queue)} of {len(REVIEW_EXPORT_FRAME)} rows selected ({REVIEW_MODE}).")
        build_review_widget(local_queue, reviewer_id=reviewer_id, on_submit=REVIEW_VALIDATIONS.append)
    else:
        print("ℹ️ HITL review skipped: no classified complete-minute rows are available.")

    if REVIEW_EXPORT_FRAME is not None:
        def finalize_current_review():
            canonical_root = None
            if IN_COLAB:
                from google.colab import drive  # type: ignore[import-untyped]
                drive.mount("/content/drive", force_remount=False)
                canonical_root = Path("/content/drive/MyDrive/vaaet-ml/data/hitl-reviews")
            result = finalize_review_session(
                classified=REVIEW_EXPORT_FRAME,
                validations=REVIEW_VALIDATIONS,
                pipeline_run_id=INFERENCE_PIPELINE_RUN_ID or LOCAL_INFERENCE_RUN_ID,
                model_version=manifest["model_version"],
                git_commit=GIT_COMMIT,
                vaaet_version=package_version("vaaet-ml"),
                local_root=REPO_ROOT / "data/processed/hitl-reviews",
                canonical_root=canonical_root,
            )
            print(f"📦 HITL package {result.package_id} | reviewed={result.reviewed_rows} | pending={result.pending_rows}")
            print(f"   fingerprint={result.fingerprint} | status={result.sync_status}")
            print(f"   location={result.canonical_path or result.local_path}")
            if result.sync_error:
                print(f"⚠️ Drive sync failed; local package remains pending: {result.sync_error}")
            return result
        print("Después de revisar u omitir filas, ejecutá finalize_current_review() una vez. Repetirlo es idempotente.")
else:
    print("ℹ️ Human review disabled. Set ENABLE_HUMAN_REVIEW=True after processing the clip.")


## 7. Visualización

El dashboard resume distribución de estados, velocidad, confianza, tipos de vehículos y relación velocidad/volumen. Sólo se ejecuta cuando `SHOW_DASHBOARD=True` y existen minutos clasificados.

In [ ]:
# Cell 7 — Optional visualization dashboard

import matplotlib.pyplot as plt

def show_dashboard(df: pd.DataFrame) -> None:
    """Display a 5-panel summary dashboard for the classified telemetry.

    Panels:
      1. Traffic state distribution (bar)
      2. Average speed over time (line)
      3. Classification confidence histogram
      4. Vehicle counts by type (stacked area)
      5. Speed vs. total vehicles scatter

    Args:
        df: Classified DataFrame with traffic_state, avg_speed, confidence,
            and per-type count columns.
    """
    fig = plt.figure(figsize=(20, 10))
    colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
    type_colors = {
        "car": "#3498db", "truck": "#e67e22", "bus": "#e74c3c",
        "motorcycle": "#2ecc71", "bicycle": "#9b59b6",
    }

    # Panel 1: State distribution
    ax1 = fig.add_subplot(2, 3, 1)
    dist = df["traffic_state"].value_counts().sort_index()
    state_names = [label_mapping.get(c, f"State {c}") for c in sorted(dist.index)]
    state_colors = [colors[c] for c in sorted(dist.index)]
    ax1.bar(state_names, dist.values, color=state_colors)
    ax1.set_title("Traffic State Distribution")
    ax1.set_ylabel("Records")

    # Panel 2: Speed timeline
    ax2 = fig.add_subplot(2, 3, 2)
    x_axis = range(len(df))
    ax2.plot(x_axis, df["avg_speed"], color="#3498db", linewidth=1.5, label="Avg Speed")
    if "speed_variance" in df.columns:
        ax2.fill_between(
            x_axis,
            df["avg_speed"] - df["speed_variance"].clip(lower=0).pow(0.5),
            df["avg_speed"] + df["speed_variance"].clip(lower=0).pow(0.5),
            alpha=0.2, color="#3498db", label="±1 σ",
        )
    ax2.set_title("Average Speed Over Time")
    ax2.set_xlabel("Minute")
    ax2.set_ylabel("Speed (km/h)")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # Panel 3: Confidence distribution
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.hist(df["confidence"], bins=20, color="#9b59b6", edgecolor="white")
    ax3.axvline(0.8, color="#e74c3c", linestyle="--", linewidth=1, label="Threshold 0.8")
    ax3.set_title("Classification Confidence")
    ax3.set_xlabel("Confidence")
    ax3.set_ylabel("Frequency")
    ax3.legend(fontsize=8)

    # Panel 4: Vehicle counts by type (stacked area)
    ax4 = fig.add_subplot(2, 3, 4)
    count_cols = [c for c in ["count_car", "count_truck", "count_bus",
                               "count_motorcycle", "count_bicycle"]
                  if c in df.columns]
    if count_cols:
        df_counts = df[count_cols].fillna(0)
        ax4.stackplot(
            x_axis, *[df_counts[c] for c in count_cols],
            labels=[c.replace("count_", "") for c in count_cols],
            colors=[type_colors.get(c.replace("count_", ""), "#999") for c in count_cols],
            alpha=0.8,
        )
        ax4.set_title("Vehicle Counts by Type")
        ax4.set_xlabel("Minute")
        ax4.set_ylabel("Count")
        ax4.legend(loc="upper left", fontsize=7)
    else:
        ax4.text(0.5, 0.5, "No count data", ha="center", va="center")
        ax4.set_title("Vehicle Counts by Type")

    # Panel 5: Speed vs. total vehicles (scatter)
    ax5 = fig.add_subplot(2, 3, 5)
    if "total_vehicles" in df.columns:
        scatter_colors = [colors[c] if c < len(colors) else "#999"
                          for c in df["traffic_state"]]
        ax5.scatter(df["total_vehicles"], df["avg_speed"], c=scatter_colors,
                    alpha=0.7, edgecolors="white", linewidth=0.5)
        ax5.set_xlabel("Total Vehicles")
        ax5.set_ylabel("Avg Speed (km/h)")
        ax5.set_title("Speed vs. Volume")
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, "No volume data", ha="center", va="center")
        ax5.set_title("Speed vs. Volume")

    # Panel 6: Summary text panel
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.axis("off")
    summary_lines = [
        f"📊 Total records: {len(df)}",
        f"⏱️  Avg speed: {df['avg_speed'].mean():.1f} km/h",
        f"📈 Max speed: {df['avg_speed'].max():.1f} km/h",
        f"📉 Min speed: {df['avg_speed'].min():.1f} km/h",
    ]
    if "total_vehicles" in df.columns:
        summary_lines.append(f"🚗 Total vehicles: {df['total_vehicles'].sum():.0f}")
    if "confidence" in df.columns:
        summary_lines.append(f"🎯 Avg confidence: {df['confidence'].mean():.3f}")
        low_conf = (df["confidence"] < 0.8).sum()
        summary_lines.append(f"⚠️  Low confidence (<0.8): {low_conf}")
    summary_text = "\n".join(summary_lines)
    ax6.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment="center",
             fontfamily="monospace", transform=ax6.transAxes)
    ax6.set_title("Summary")

    plt.tight_layout()
    plt.show()


# Execution — requires df_classified from the analysis cell
try:
    if not SHOW_DASHBOARD:
        print("ℹ️ Dashboard disabled by the central configuration.")
    elif df_classified is not None and not df_classified.empty:
        show_dashboard(df_classified)
    else:
        print("⚠️ No classified data — stable inference requires at least two consecutive complete minutes.")
except NameError:
    print("⚠️ df_classified is not defined — run the video analysis cell first.")
except Exception as e:
    print(f"🔴 Dashboard error: {e}")